In [ ]:
# Code pour créer les fichiers excels pour la publicité 
import pandas as pd
from pathlib import Path
from io import StringIO
import re
from unidecode import unidecode

def nettoyer_nom(nom):
    return re.sub(r'[\\/*?:"<>|]', "_", str(nom))

path_sauv = Path(r"C:\Users\L14\Desktop\sauvegardes_digifor")
path_enquete = r"C:\Users\L14\Downloads\listes_des_enquetes_tonkpi_zh_sip_dan_biank_man_vf1.csv"

files = path_sauv.rglob('*.csv')
bd_enquetes = pd.read_csv(path_enquete,encoding='utf-8',sep=';',encoding_errors='replace')

for file in files:
    with open(file, "r", encoding='utf-8') as f:
        lines = f.readlines()

        tables = {}
        current_table_name = None
        current_table_lines = []
        tables_titles = []
        for line in lines:
            line = line.strip()

            if line == "":
                continue

            if line.lower().startswith("table:"):
                
                if current_table_name and current_table_lines:
                    csv_block = "\n".join(current_table_lines)
                    df = pd.read_csv(StringIO(csv_block))
                    tables[current_table_name] = df
                    current_table_lines = []
                
                current_table_name = line.split(":", 1)[1].strip() 
                tables_titles.append(current_table_name)
            else:
                current_table_lines.append(line)

            
        if current_table_name and current_table_lines:
            csv_block = "\n".join(current_table_lines)
            df = pd.read_csv(StringIO(csv_block))
            tables[current_table_name] = df
        
        df_applicant = tables['applicant'][['id','applicantNumber','firstname','lastname','habitualResidence','typeOfIndividualCertificate','numParcelleOF']].copy()
        df_applicant.loc[:,'applicantNumber'] = df_applicant['applicantNumber'].str.replace('APT','')

        df_land = tables['land'][['id','code','label','estimatedArea','description','village']].copy()
        df_request = tables['request'][['id','requestNumber','nomPrenomCE','applicant','land']].copy()
        df_land.loc[:,'code'] = df_land['code'].str.replace('LAN','')
        df_merge_request_applicant = pd.merge(df_request,df_applicant,left_on='requestNumber',right_on='applicantNumber',how='inner')
        df_merge_request__applicant_land = pd.merge(df_merge_request_applicant,df_land,left_on='requestNumber',right_on='code',how='inner')
        df_neighborhood = tables['neighborhood'][['code','position','nameOfNeighbor']].copy()
        df_neighborhood.loc[:,'code'] = df_neighborhood['code'].str.replace('NEI','')
        df_neighborhood.loc[:,'nameOfNeighbor'] = "Nom : " + df_neighborhood['nameOfNeighbor']
        df_neighborhood.loc[:,'position'] = "Position : " + df_neighborhood['position']
        df_neighborhood.loc[:,'VOISINS ET ORIENTATIONS'] = df_neighborhood['nameOfNeighbor'] + " , " + df_neighborhood['position']
        df_neighborhood.loc[:,'VOISINS ET ORIENTATIONS'] = df_neighborhood['VOISINS ET ORIENTATIONS'].apply(lambda x: str(x))
        df_neighborhood_grouped = df_neighborhood.groupby('code').agg({
        'VOISINS ET ORIENTATIONS': lambda x: ' | '.join(x)
        }).reset_index()

        df_merge_applicant_neighborhood_grouped = pd.merge(df_neighborhood_grouped,df_merge_request__applicant_land,left_on='code',right_on='applicantNumber',how='inner')
        df_merge_enquetes = pd.merge(df_merge_applicant_neighborhood_grouped,bd_enquetes,left_on='code_x',right_on='N° de la demande',how='inner')
        
        df_merge_enquetes.rename(columns={
        'N° de la demande':'NUMERO DE DEMANDE',
        'typeOfIndividualCertificate':'TYPE DE CERTIFICAT',
        'Numero Parcelle':'CODE PARCELLE'
        },inplace=True)
        
        df_merge_enquetes = df_merge_enquetes[['NUMERO DE DEMANDE','NOM ET PRENOMS','CODE PARCELLE','TYPE DE CERTIFICAT','SUPERFICIE (Hectares)','VOISINS ET ORIENTATIONS','description','village','nomPrenomCE']].copy()

        villages = df_merge_enquetes['village'].unique()

        for item in villages:

            try:
                village = df_merge_enquetes[df_merge_enquetes['village'] == item]
                sous_prefecture = village['description'].iloc[0]
                agent = '_'.join(str(file).split('\\')[-1].split('.')[0].split('_')[2:])
                nom_village = nettoyer_nom(item)
                destination = fr"C:\Users\L14\Desktop\demandes_sauvegardes_digifor\{sous_prefecture.lower()}_{agent}_{nom_village}.csv"
                destination_excel = fr"C:\Users\L14\Desktop\new_demandes_sauvegardes_digifor\{sous_prefecture}_{agent}_{nom_village}.xlsx"
                village = village.copy()
                village['VOISINS ET ORIENTATIONS'] = village['VOISINS ET ORIENTATIONS'].apply(lambda x : x.replace('|','\n').replace(',',';'))
                village[['NUMERO DE DEMANDE','NOM ET PRENOMS','CODE PARCELLE','TYPE DE CERTIFICAT','SUPERFICIE (Hectares)','VOISINS ET ORIENTATIONS']].to_excel(destination_excel,index=False)
            except:
                print(f' Error creating {sous_prefecture}_{nom_village}')
            else:
                print(f'{sous_prefecture}_{nom_village} created succeed')




In [ ]:
# Code pour faire le bilan des publicités par village
import pandas as pd
import re
from io import StringIO

path_enquete = r"C:\Users\L14\Downloads\listes_des_enquetes_tonkpi_zh_sip_dan_biank_man_vf1.csv"

enquetes = pd.read_csv(path_enquete,encoding='utf-8',sep=';',encoding_errors='replace')
print(enquetes.columns)

enquetes['SUPERFICIE (Hectares)'] = pd.to_numeric(enquetes['SUPERFICIE (Hectares)'])
bilan_pub = enquetes.groupby('ID village').agg(
        total_superficie=('SUPERFICIE (Hectares)','sum'),
        total_parcelle=('N° de la demande','count')
        ).reset_index()

bilan_pub.to_excel(r"C:\Users\L14\Downloads\bilan_pub_vf.xlsx",index=False)

In [ ]:
# Code pour transformer les sauvegardes de digifor en fichier excel
import pandas as pd
from pathlib import Path
from io import StringIO
import re
from unidecode import unidecode

def nettoyer_nom(nom):
    return re.sub(r'[\\/*?:"<>|]', "_", str(nom))

path_sauv = Path(r"C:\Users\L14\Desktop\sauvegardes_digifor")

files = path_sauv.rglob('*.csv')

for file in files:
    #print(file)
    with open(file, "r", encoding='utf-8') as f:
        lines = f.readlines()

        tables = {}
        current_table_name = None
        current_table_lines = []
        tables_titles = []
        for line in lines:
            line = line.strip()

            if line == "":
                continue

            if line.lower().startswith("table:"):
                
                if current_table_name and current_table_lines:
                    csv_block = "\n".join(current_table_lines)
                    df = pd.read_csv(StringIO(csv_block))
                    tables[current_table_name] = df
                    current_table_lines = []
                
                current_table_name = line.split(":", 1)[1].strip() 
                tables_titles.append(current_table_name)
            else:
                current_table_lines.append(line)

        if current_table_name and current_table_lines:
            csv_block = "\n".join(current_table_lines)
            df = pd.read_csv(StringIO(csv_block))
            tables[current_table_name] = df

        df_applicant = tables['applicant'][['id','applicantNumber','firstname','lastname','habitualResidence','typeOfIndividualCertificate','numParcelleOF']].copy()
        df_applicant.loc[:,'applicantNumber'] = df_applicant['applicantNumber'].str.replace('APT','')

        df_land = tables['land'][['id','code','label','estimatedArea','description','village']].copy()
        df_request = tables['request'][['id','requestNumber','nomPrenomCE','applicant','land']].copy()
        df_land.loc[:,'code'] = df_land['code'].str.replace('LAN','')
        df_merge_request_applicant = pd.merge(df_request,df_applicant,left_on='requestNumber',right_on='applicantNumber',how='inner')
        df_merge_request__applicant_land = pd.merge(df_merge_request_applicant,df_land,left_on='requestNumber',right_on='code',how='inner')

        try:
            df_merge_request__applicant_land[['requestNumber','firstname', 'lastname','description','label', 'village','typeOfIndividualCertificate']].to_excel(fr'{str(file).split('.')[0]}.xlsx',index=False)
        except:
                print(f' Error creating {str(file).split('.')[0]}')
        else:
            print(f'{str(file)} created succeed')